# PyTorch DDP Speech Recognition with Kubeflow Trainer

Distributed training of Audio Transformer for speech command classification using the Google Speech Commands dataset.

## Features

- Distributed Data Parallel (DDP) training support
- Memory-optimized implementation
- Cross-platform compatibility (Windows/Linux/Mac)
- TensorBoard logging and model checkpointing

## Installation

Install required packages for training. This includes Kubeflow Training SDK, PyTorch, audio processing libraries, and TensorBoard for visualization.

In [8]:
import subprocess
import sys

packages = [
    'kubeflow-training',
    'torch',
    'torchaudio',
    'librosa',
    'soundfile',
    'tensorboard'
]

print("Installing packages...")
for package in packages:
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', package, '--no-cache-dir', '-q'],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print(f"Installed: {package}")
        else:
            print(f"Failed to install {package}: {result.stderr}")
    except Exception as e:
        print(f"Error installing {package}: {e}")

print("\nInstallation complete")

Installing packages...
Installed: kubeflow-training
Installed: torch
Installed: torchaudio
Installed: librosa
Installed: soundfile
Installed: tensorboard

Installation complete


## Setup

Configure the data directory based on the operating system. Windows uses C:/data while Unix-like systems use ~/data.

In [9]:
import os
from pathlib import Path
import platform

# Configure data directory based on operating system
if platform.system() == 'Windows':
    DATA_DIR = Path("C:/data")
else:
    DATA_DIR = Path.home() / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data directory: {DATA_DIR.absolute()}")

Data directory: C:\data


## Training Function

The training function implements distributed data parallel training for speech recognition.
Key components:
- DDP initialization with NCCL (GPU) or Gloo (CPU) backend
- Google Speech Commands dataset loading and preprocessing
- Audio Transformer model architecture
- Training loop with validation and checkpointing
- TensorBoard logging for monitoring

In [10]:
def train_fn():
    """DDP training function for speech recognition."""
    import torch
    import torch.nn as nn
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import Dataset, DataLoader, DistributedSampler
    import torchaudio
    from torch.utils.tensorboard import SummaryWriter
    import os
    import random
    import gc
    from datetime import datetime
    from pathlib import Path
    import platform

    # Configuration - supports debug mode for faster testing
    debug = os.environ.get("DEBUG_MODE", "true").lower() == "true"
    data_path_str = os.environ.get("DATA_PATH")
    
    if not data_path_str:
        data_path = Path("C:/data") if platform.system() == 'Windows' else Path.home() / "data"
    else:
        data_path = Path(data_path_str)
    
    print(f"Training mode: {'DEBUG' if debug else 'FULL'}")
    print(f"Data path: {data_path}")

    def setup_ddp():
        """
        Initialize distributed training environment.
        
        Checks for DDP environment variables (LOCAL_RANK, RANK, WORLD_SIZE).
        Selects appropriate backend: NCCL for GPU, Gloo for CPU.
        Falls back to single-process mode if DDP is not available.
        
        Returns:
            tuple: (device, local_rank, rank)
        """
        if all(k in os.environ for k in ["LOCAL_RANK", "RANK", "WORLD_SIZE"]):
            local_rank = int(os.environ["LOCAL_RANK"])
            rank = int(os.environ["RANK"])
            world_size = int(os.environ["WORLD_SIZE"])
            
            print(f"Initializing DDP: rank {rank}/{world_size}")
            
            if torch.cuda.is_available():
                backend = "nccl"
                device = torch.device("cuda", local_rank)
                torch.cuda.set_device(device)
            else:
                backend = "gloo"
                device = torch.device("cpu")
            
            try:
                dist.init_process_group(backend=backend)
                print(f"DDP initialized: {backend}")
            except Exception as e:
                print(f"DDP initialization failed: {e}")
                print("Falling back to single-process mode")
                device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                return device, 0, 0
            
            return device, local_rank, rank
        else:
            print("Single-process mode")
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            return device, 0, 0

    def cleanup_ddp():
        """
        Cleanup distributed resources.
        
        Destroys the process group if DDP was initialized.
        Should be called at the end of training or on error.
        """
        if dist.is_initialized():
            try:
                dist.destroy_process_group()
            except Exception as e:
                print(f"Cleanup error: {e}")

    def load_data(data_root):
        """
        Load audio file metadata and create label mapping.
        
        Scans the dataset directory for label subdirectories and audio files.
        Creates a mapping from label names to integer indices.
        
        Args:
            data_root: Path to the dataset root directory
            
        Returns:
            tuple: (audio_info, label_map) where audio_info is a list of dicts
                   containing filename and label, and label_map maps labels to indices
        """
        audio_info = []
        data_root = Path(data_root)
        
        if not data_root.exists():
            raise FileNotFoundError(f"Data directory not found: {data_root}")
        
        # Get all label directories (excluding internal directories starting with _)
        labels = sorted([
            item.name for item in data_root.iterdir()
            if item.is_dir() and not item.name.startswith("_")
        ])
        
        if not labels:
            raise ValueError(f"No label directories found in {data_root}")
        
        # Create label to index mapping
        label_map = {label: idx for idx, label in enumerate(labels)}
        
        # Collect all audio files with their labels
        for label in labels:
            for audio_file in (data_root / label).glob("*.wav"):
                audio_info.append({"filename": str(audio_file), "label": label})
        
        if not audio_info:
            raise ValueError(f"No audio files found in {data_root}")
        
        print(f"Loaded {len(audio_info)} files, {len(labels)} classes")
        return audio_info, label_map

    def split_data(audio_info):
        """
        Split dataset into train/validation/test sets.
        
        Uses a 95/3/2 split ratio for training, validation, and testing.
        Shuffles data with a fixed seed for reproducibility.
        
        Args:
            audio_info: List of audio file information dictionaries
            
        Returns:
            tuple: (train_data, val_data, test_data)
        """
        random.seed(42)
        random.shuffle(audio_info)
        n = len(audio_info)
        train_size = int(n * 0.95)
        val_size = int(n * 0.03)
        return (
            audio_info[:train_size],
            audio_info[train_size:train_size + val_size],
            audio_info[train_size + val_size:]
        )

    class SpeechDataset(Dataset):
        """
        Speech command dataset with mel spectrogram features.
        
        Converts raw audio waveforms to mel spectrograms, which are more suitable
        for training neural networks on audio data. Handles variable-length audio
        by padding or truncating to a fixed length.
        """
        def __init__(self, audio_info, label_map):
            self.audio_info = audio_info
            self.label_map = label_map
            # Mel spectrogram parameters optimized for speech
            self.transform = torchaudio.transforms.MelSpectrogram(
                n_mels=128,
                n_fft=1024,
                hop_length=512
            )
            self.target_length = 81

        def __len__(self):
            return len(self.audio_info)

        def __getitem__(self, idx):
            """
            Load and preprocess a single audio sample.
            
            Steps:
            1. Load audio waveform
            2. Convert stereo to mono if needed
            3. Compute mel spectrogram
            4. Pad or truncate to fixed length
            5. Return spectrogram and label
            """
            try:
                info = self.audio_info[idx]
                waveform, _ = torchaudio.load(info["filename"])
                
                # Convert stereo to mono by averaging channels
                if waveform.shape[0] > 1:
                    waveform = torch.mean(waveform, dim=0, keepdim=True)
                
                # Compute mel spectrogram
                spec = self.transform(waveform)
                
                # Pad or truncate to fixed length for batching
                current_len = spec.shape[2]
                if current_len < self.target_length:
                    spec = torch.nn.functional.pad(spec, (0, self.target_length - current_len))
                else:
                    spec = spec[:, :, :self.target_length]
                
                label = self.label_map[info["label"]]
                del waveform
                return spec.squeeze(0), label
            except Exception as e:
                print(f"Error loading {info.get('filename', 'unknown')}: {e}")
                # Return zero tensor on error to avoid crashing the training
                return torch.zeros(128, self.target_length), 0

    def collate_fn(batch):
        """
        Collate batch samples into tensors.
        
        Stacks spectrograms and labels from individual samples into batch tensors.
        """
        specs, labels = zip(*batch)
        return torch.stack(specs), torch.tensor(labels, dtype=torch.long)

    class AudioTransformer(nn.Module):
        """
        Transformer-based audio classifier.
        
        Uses a transformer encoder to process mel spectrogram features.
        The temporal dimension of the spectrogram is treated as the sequence length.
        Global average pooling is applied before classification.
        """
        def __init__(self, num_classes=35, d_model=128, nhead=4, num_layers=2):
            super().__init__()
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=512,
                batch_first=True,
                dropout=0.1
            )
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.classifier = nn.Linear(d_model, num_classes)

        def forward(self, x):
            """
            Forward pass through the model.
            
            Args:
                x: Input tensor of shape (batch, n_mels, time)
                
            Returns:
                Output logits of shape (batch, num_classes)
            """
            # Permute to (batch, time, n_mels) for transformer
            x = x.permute(0, 2, 1)
            # Apply transformer encoder
            x = self.transformer(x)
            # Global average pooling over time dimension
            x = x.mean(dim=1)
            # Classification layer
            return self.classifier(x)

    class Trainer:
        """
        Training manager with checkpointing and logging.
        
        Handles the training loop, validation, model checkpointing,
        and TensorBoard logging. Only rank 0 performs logging and saving.
        """
        def __init__(self, model, train_loader, val_loader, optimizer, criterion, device, rank, epochs):
            self.model = model
            self.train_loader = train_loader
            self.val_loader = val_loader
            self.optimizer = optimizer
            self.criterion = criterion
            self.device = device
            self.rank = rank
            self.epochs = epochs
            self.is_main = (rank == 0)
            self.best_acc = 0.0
            
            # Initialize experiment directory and TensorBoard on main process
            if self.is_main:
                timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
                self.exp_path = data_path / "runs" / f"exp-{timestamp}"
                self.exp_path.mkdir(parents=True, exist_ok=True)
                (self.exp_path / "logs").mkdir(exist_ok=True)
                (self.exp_path / "models").mkdir(exist_ok=True)
                self.writer = SummaryWriter(str(self.exp_path / "logs"))
                print(f"Experiment: {self.exp_path}")
            else:
                self.writer = None

        def train_epoch(self, epoch):
            """
            Train for one epoch.
            
            Iterates through the training data, performs forward and backward passes,
            and updates model parameters. Logs training loss periodically.
            """
            self.model.train()
            # Set epoch for DistributedSampler to ensure proper shuffling
            if hasattr(self.train_loader.sampler, 'set_epoch'):
                self.train_loader.sampler.set_epoch(epoch)
            
            total_loss = 0
            num_batches = 0
            
            for batch_idx, (data, target) in enumerate(self.train_loader):
                data, target = data.to(self.device), target.to(self.device)
                
                # Standard training step
                self.optimizer.zero_grad()
                output = self.model(data)
                loss = self.criterion(output, target)
                loss.backward()
                
                # Gradient clipping to prevent exploding gradients
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                
                self.optimizer.step()
                total_loss += loss.item()
                num_batches += 1
                
                # Log progress periodically
                if self.is_main and batch_idx % 10 == 0:
                    print(f"Epoch {epoch+1}/{self.epochs} | Batch {batch_idx}/{len(self.train_loader)} | Loss: {loss.item():.4f}")
                
                # Clean up to free memory
                del data, target, output, loss
                if batch_idx % 50 == 0:
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
            
            # Log average epoch loss
            if self.is_main and num_batches > 0:
                avg_loss = total_loss / num_batches
                self.writer.add_scalar("Loss/train", avg_loss, epoch)
                print(f"Epoch {epoch+1} avg loss: {avg_loss:.4f}")

        def validate(self, epoch):
            """
            Validate model and save best checkpoint.
            
            Evaluates the model on the validation set without gradient computation.
            Saves the model if it achieves a new best accuracy.
            """
            if not self.is_main or self.val_loader is None:
                return
            
            self.model.eval()
            correct = total = 0
            
            with torch.no_grad():
                for data, target in self.val_loader:
                    data, target = data.to(self.device), target.to(self.device)
                    output = self.model(data)
                    pred = output.argmax(dim=1)
                    correct += (pred == target).sum().item()
                    total += target.size(0)
            
            accuracy = 100.0 * correct / total if total > 0 else 0.0
            self.writer.add_scalar("Accuracy/val", accuracy, epoch)
            print(f"Validation accuracy: {accuracy:.2f}%")
            
            # Save model if it's the best so far
            if accuracy > self.best_acc:
                self.best_acc = accuracy
                # Extract state dict from DDP wrapper if needed
                state = self.model.module.state_dict() if hasattr(self.model, 'module') else self.model.state_dict()
                model_path = self.exp_path / "models" / f"best_model_epoch{epoch+1}_acc{accuracy:.2f}.pth"
                torch.save(state, str(model_path))
                print(f"Saved best model: {accuracy:.2f}%")

        def train(self):
            """
            Run the full training loop.
            
            Iterates through epochs, training and validating the model.
            Synchronizes processes with barriers in distributed mode.
            """
            for epoch in range(self.epochs):
                print(f"\nEpoch {epoch+1}/{self.epochs}")
                
                self.train_epoch(epoch)
                self.validate(epoch)
                
                # Synchronize all processes before next epoch
                if dist.is_initialized():
                    dist.barrier()
            
            if self.is_main:
                self.writer.close()
                print(f"\nTraining complete. Best accuracy: {self.best_acc:.2f}%")

    # Main training logic
    try:
        device, local_rank, rank = setup_ddp()
        print(f"Device: {device}")
        
        # Load dataset
        data_root = data_path / "SpeechCommands" / "speech_commands_v0.02"
        audio_info, label_map = load_data(data_root)
        train_data, val_data, test_data = split_data(audio_info)
        
        # Use subset for debug mode
        if debug:
            train_data = train_data[:500]
            val_data = val_data[:100]
        
        print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
        
        # Create datasets
        train_dataset = SpeechDataset(train_data, label_map)
        val_dataset = SpeechDataset(val_data, label_map)
        
        # Setup data loaders with distributed sampler if using DDP
        train_sampler = DistributedSampler(train_dataset) if dist.is_initialized() else None
        batch_size = 16 if debug else 32
        
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=train_sampler,
            shuffle=(train_sampler is None),
            collate_fn=collate_fn,
            num_workers=0,
            pin_memory=(device.type == 'cuda')
        )
        
        # Only rank 0 needs validation loader
        val_loader = None
        if rank == 0:
            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_fn,
                num_workers=0
            )
        
        # Initialize model and wrap with DDP if needed
        model = AudioTransformer(num_classes=len(label_map)).to(device)
        
        if dist.is_initialized():
            model = DDP(model, device_ids=[local_rank] if torch.cuda.is_available() else None)
        
        # Setup optimizer and loss function
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        epochs = 3 if debug else 10
        
        print(f"Training: {epochs} epochs, batch size {batch_size}")
        
        # Run training
        trainer = Trainer(model, train_loader, val_loader, optimizer, criterion, device, rank, epochs)
        trainer.train()
        
        # Cleanup
        cleanup_ddp()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        print("\nTraining completed successfully")
        
    except Exception as e:
        print(f"\nTraining failed: {e}")
        import traceback
        traceback.print_exc()
        cleanup_ddp()
        raise

print("Training function defined")

Training function defined


## Download Dataset

Downloads the Google Speech Commands v0.02 dataset (approximately 2.3GB). This should only be run once. The dataset contains short audio clips of spoken commands like "yes", "no", "up", "down", etc.

In [11]:
import torchaudio

dataset_path = DATA_DIR / "SpeechCommands" / "speech_commands_v0.02"

if dataset_path.exists():
    total_files = sum(1 for _ in dataset_path.rglob("*.wav"))
    print(f"Dataset exists: {total_files} files")
    if total_files < 1000:
        print("Warning: Dataset may be incomplete")
else:
    print(f"Downloading dataset to {DATA_DIR}")
    print("This may take 5-10 minutes...")
    try:
        dataset = torchaudio.datasets.SPEECHCOMMANDS(root=str(DATA_DIR), download=True)
        print(f"Download complete: {len(dataset)} samples")
    except Exception as e:
        print(f"Download failed: {e}")
        print("\nManual download: http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz")
        raise

Dataset exists: 105835 files


## Run Training

Execute the training function. Set DEBUG_MODE to "true" for quick testing with a subset of data, or "false" for full training.

In [12]:
# Set to "false" for full training
os.environ["DEBUG_MODE"] = "true"
os.environ["DATA_PATH"] = str(DATA_DIR)

print("Starting training")
print()

try:
    train_fn()
    print("\nTraining completed")
except FileNotFoundError as e:
    print(f"\nData not found: {e}")
    print("Please run the dataset download cell first")
except Exception as e:
    print(f"\nTraining failed: {e}")
    import traceback
    traceback.print_exc()

Starting training

Training mode: DEBUG
Data path: C:\data
Single-process mode
Device: cpu
Loaded 105829 files, 35 classes
Train: 500, Val: 100, Test: 2118
Training: 3 epochs, batch size 16
Experiment: C:\data\runs\exp-20260205-100700

Epoch 1/3
Error loading C:\data\SpeechCommands\speech_commands_v0.02\yes\7d6b4b10_nohash_2.wav: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8. On Windows, ensure you've installed
             the "full-shared" version which ships DLLs.
          2. The PyTorch version (2.9.1+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.
        The following exceptions were raised as we tried to load libtorchcodec:
        
[s

## View Results

Check the latest experiment directory for TensorBoard logs and saved model checkpoints.

In [13]:
runs_dir = DATA_DIR / "runs"

if runs_dir.exists():
    experiments = sorted(runs_dir.glob("exp-*"), key=lambda p: p.stat().st_mtime, reverse=True)
    if experiments:
        latest_exp = experiments[0]
        print(f"Latest experiment: {latest_exp}")
        print(f"\nTensorBoard: tensorboard --logdir {latest_exp / 'logs'}")
        
        models_dir = latest_exp / "models"
        if models_dir.exists():
            models = list(models_dir.glob("*.pth"))
            print(f"\nSaved models ({len(models)}):")
            for model_path in models:
                print(f"  {model_path.name}")
        
        print(f"\nAll experiments ({len(experiments)}):")
        for exp in experiments[:5]:
            print(f"  {exp.name}")
    else:
        print("No experiments found")
else:
    print("No runs directory found")

Latest experiment: C:\data\runs\exp-20260205-100700

TensorBoard: tensorboard --logdir C:\data\runs\exp-20260205-100700\logs

Saved models (1):
  best_model_epoch1_acc100.00.pth

All experiments (9):
  exp-20260205-100700
  exp-20260205-093807
  exp-20260205-092408
  exp-20260205-092334
  exp-20260205-091633


## Load Trained Model

Instructions for loading a saved model checkpoint for inference or further training.

In [14]:
import torch

runs_dir = DATA_DIR / "runs"
if runs_dir.exists():
    experiments = sorted(runs_dir.glob("exp-*"), key=lambda p: p.stat().st_mtime, reverse=True)
    if experiments:
        latest_exp = experiments[0]
        models_dir = latest_exp / "models"
        if models_dir.exists():
            models = list(models_dir.glob("*.pth"))
            if models:
                model_path = models[0]
                print(f"Model: {model_path.name}")
                print("\nTo load:")
                print("1. Recreate AudioTransformer architecture")
                print("2. model.load_state_dict(torch.load(model_path))")
                print("3. model.eval()")
            else:
                print("No models found")
        else:
            print("Models directory not found")
    else:
        print("No experiments found")
else:
    print("No runs directory found")

Model: best_model_epoch1_acc100.00.pth

To load:
1. Recreate AudioTransformer architecture
2. model.load_state_dict(torch.load(model_path))
3. model.eval()
